# 01_checkers — All result_* checkers and their parameters

Every `@regular_aspect` publishes a typed contract via `@result_*` decorators.
This notebook shows all six checker types in one pipeline step.

| Checker | Field type | Extra parameters |
|---------|-----------|-----------------|
| `result_string` | `str` | `min_length`, `max_length`, `not_empty` |
| `result_int` | `int` | `min_value`, `max_value` |
| `result_float` | `float` | `min_value`, `max_value` |
| `result_bool` | `bool` | — |
| `result_date` | `str` or `datetime` | `date_format`, `min_date`, `max_date` |
| `result_instance` | class instance | `no_none`, `value_check` |

All checkers share `required=True/False` and `opaque=True/False` (see 07_opaque).

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio
from dataclasses import dataclass
from datetime import datetime

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import (
    result_bool,
    result_date,
    result_float,
    result_instance,
    result_int,
    result_string,
)
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain and supporting types

In [ ]:
class OrderDomain(BaseDomain):
    name = "order"
    description = "Order intake and validation domain"


@dataclass
class CustomerProfile:
    name: str
    is_active: bool

## Params and Result

In [ ]:
class OrderParams(BaseParams):
    order_id: str = Field(description="Raw order identifier from the client")
    quantity: int = Field(description="Number of units")
    unit_price: float = Field(description="Price per unit in USD")
    is_urgent: bool = Field(description="Whether express processing is requested")
    delivery_date: str = Field(description="Requested delivery date (YYYY-MM-DD)")
    customer_name: str = Field(description="Full name of the customer")
    coupon_code: str | None = Field(default=None, description="Optional discount coupon")


class OrderResult(BaseResult):
    order_id: str = Field(description="Normalised order ID (uppercased)")
    total: float = Field(description="Computed order total")
    is_urgent: bool = Field(description="Express processing flag")
    has_coupon: bool = Field(description="Whether a coupon was applied")

## Action — all six checker types on one aspect

- **`result_string("order_id", min_length=5, max_length=20)`** — string between 5 and 20 characters
- **`result_int("quantity", min_value=1, max_value=999)`** — integer in `[1, 999]`, both bounds inclusive
- **`result_float("unit_price", min_value=0.01)`** — float ≥ 0.01, no upper bound
- **`result_bool("is_urgent")`** — boolean, no extra params beyond `required`
- **`result_date("delivery_date", date_format="%Y-%m-%d", min_date=...)`** — string parsed with `strptime`, must be after 2024-01-01
- **`result_instance("customer", CustomerProfile, no_none=True, value_check=...)`** — instance of `CustomerProfile`; `no_none=True` rejects explicit `None`; `value_check` lambda must return `True`
- **`result_string("coupon_code", required=False)`** — optional: absent from returned dict → silently skipped

In [ ]:
@meta(description="Validate and normalise an incoming order", domain=OrderDomain)
@check_roles(GuestRole)
class ValidateOrderAction(BaseAction[OrderParams, OrderResult]):

    @regular_aspect("Validate and normalise all order fields")
    @result_string("order_id", required=True, min_length=5, max_length=20)
    @result_int("quantity", required=True, min_value=1, max_value=999)
    @result_float("unit_price", required=True, min_value=0.01)
    @result_bool("is_urgent", required=True)
    @result_date("delivery_date", required=True, date_format="%Y-%m-%d",
                 min_date=datetime(2024, 1, 1))
    @result_instance("customer", CustomerProfile, required=True, no_none=True,
                     value_check=lambda c: c.is_active)
    @result_string("coupon_code", required=False)
    async def validate_aspect(self, params, state, box, connections):
        result: dict = {
            "order_id": params.order_id.strip().upper(),
            "quantity": params.quantity,
            "unit_price": params.unit_price,
            "is_urgent": params.is_urgent,
            "delivery_date": params.delivery_date,
            "customer": CustomerProfile(name=params.customer_name, is_active=True),
        }
        if params.coupon_code:
            result["coupon_code"] = params.coupon_code.strip().upper()
        return result

    @summary_aspect("Assemble validated order result")
    async def assemble_summary(self, params, state, box, connections):
        total = state["quantity"] * state["unit_price"]
        has_coupon = "coupon_code" in state
        return OrderResult(
            order_id=state["order_id"],
            total=round(total, 2),
            is_urgent=state["is_urgent"],
            has_coupon=has_coupon,
        )

## Run — with coupon and without

Run twice to see `required=False` in action: the second call omits `coupon_code`,
the checker is silently skipped, and `has_coupon` is `False`.

In [ ]:
machine = ActionProductMachine()

# Run 1: with coupon
result = await machine.run(
    Context(),
    ValidateOrderAction(),
    OrderParams(
        order_id="  ord-2024-001  ", quantity=3, unit_price=49.99,
        is_urgent=True, delivery_date="2025-12-01",
        customer_name="Alice", coupon_code="  summer20  ",
    ),
)
print(f"Run 1 → order_id={result.order_id!r}, total={result.total}, has_coupon={result.has_coupon}")

# Run 2: without coupon — required=False → no error
result = await machine.run(
    Context(),
    ValidateOrderAction(),
    OrderParams(
        order_id="ord-2024-002", quantity=1, unit_price=9.99,
        is_urgent=False, delivery_date="2025-06-15",
        customer_name="Bob", coupon_code=None,
    ),
)
print(f"Run 2 → order_id={result.order_id!r}, total={result.total}, has_coupon={result.has_coupon}")